In [ ]:
%pip install pandas tqdm pythainlp rank_bm25 langchain langchain-community tenacity sentence-transformers faiss-cpu openai langchain-text-splitters langchain-huggingface

In [ ]:
# =========================================
# Configuration & Credentials
# =========================================
import os

API_KEY = "your-api-key-here"
BASE_URL = "http://thaillm.or.th/api/pathumma/v1/chat/completions"

# ตั้งค่า Path ของข้อมูล
BASE_PATH = "."
QUESTIONS_PATH = "./data/questions.csv"
OUTPUT_PATH = "./submission.csv"

print("✅ ตั้งค่า Configuration และ API Key เรียบร้อย!")

### ฟังก์ชันหลักของระบบ RAG (Retrieval Augmented Generation)
เซลล์นี้ประกอบด้วยฟังก์ชันสำคัญสำหรับการโหลดเอกสาร, หั่นเอกสารเป็นส่วนย่อย (Chunking), การสร้างระบบค้นหาแบบ Hybrid (BM25 + FAISS), และการสร้าง Prompt สำหรับ LLM เพื่อให้ได้คำตอบที่แม่นยำ

In [ ]:
# =========================================
# Core RAG System
# =========================================
import os
import re
import pandas as pd
from tqdm import tqdm
from typing import List
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

# Document Loader (หั่นชิ้นใหญ่ขึ้น ป้องกันตารางขาด)
def load_markdown_documents(base_path: str) -> List[Document]:
    print(f"\n[LOG] เริ่มค้นหาเอกสารใน: {base_path}")
    all_docs = []
    headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=500,
        separators=["\n\n", "\n", " ", ""]
    )

    file_paths = []
    for root, _, files in os.walk(base_path):
        for file_name in files:
            if file_name.endswith(".md"):
                category = os.path.basename(root)
                if category == os.path.basename(base_path) or category == "":
                    category = "general"
                full_path = os.path.join(root, file_name)
                file_paths.append((category, file_name, full_path))

    for category, file_name, file_path in tqdm(file_paths, desc="📚 [LOG] กำลังอ่านไฟล์และหั่น Chunk"):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        md_docs = markdown_splitter.split_text(text)
        for md_doc in md_docs:
            header_context = " > ".join([v for k, v in md_doc.metadata.items()])
            context_prefix = f"[หมวดหมู่: {category} | ไฟล์: {file_name} | หัวข้อ: {header_context}]\n"

            chunks = text_splitter.split_text(md_doc.page_content)
            for chunk in chunks:
                new_doc = Document(
                    page_content=context_prefix + chunk,
                    metadata={"category": category, "source_file": file_name}
                )
                all_docs.append(new_doc)

    print(f"[LOG] หั่นเอกสารเสร็จสิ้น ได้ทั้งหมด: {len(all_docs)} Chunks")
    return all_docs

# Hybrid Retriever (ดึงข้อมูลแบบ Deep Search)
class HybridRetriever:
    def __init__(self, documents: List[Document], embedding_model: str = "intfloat/multilingual-e5-large", reranker_model: str = "BAAI/bge-reranker-v2-m3"):
        self.documents = documents
        self.texts = [d.page_content for d in documents]

        print(f"\n[LOG] กำลังเชื่อมต่อ HuggingFace Embeddings ({embedding_model})...")
        self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model, model_kwargs={'device': 'cuda'})

        print("\n[LOG] กำลังสร้าง FAISS Vector Store...")
        batch_size = 50
        self.faiss_index = None
        for i in tqdm(range(0, len(self.documents), batch_size), desc="🧠 [LOG] กำลังแปลง Vector (FAISS)"):
            batch_docs = self.documents[i:i+batch_size]
            if self.faiss_index is None:
                self.faiss_index = FAISS.from_documents(batch_docs, self.embeddings)
            else:
                self.faiss_index.add_documents(batch_docs)

        print("\n[LOG] กำลังเตรียมระบบค้นหาด้วย Keyword (BM25)...")
        tokenized_texts = [word_tokenize(t, engine='newmm') for t in tqdm(self.texts, desc="🔠 [LOG] กำลังตัดคำภาษาไทยทำ BM25")]
        self.bm25 = BM25Okapi(tokenized_texts)

        print(f"\n[LOG] กำลังโหลด Re-ranker Model ({reranker_model}) บน CUDA...")
        self.reranker = CrossEncoder(reranker_model, max_length=512, device='cuda')

    def retrieve(self, query: str, top_k: int = 10) -> List[Document]:
        query_for_embed = f"query: {query}"

        dense_results = self.faiss_index.similarity_search(query_for_embed, k=40)

        query_tokens = word_tokenize(query, engine='newmm')
        bm25_scores = self.bm25.get_scores(query_tokens)
        bm25_ranked_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:40]
        sparse_results = [self.documents[i] for i in bm25_ranked_indices]

        # รวมร่างด้วย RRF
        fused_docs = self.reciprocal_rank_fusion(dense_results, sparse_results, k=60)

        # คัดมาแค่ 30 อันดับแรกเพื่อส่งเข้า Re-ranker
        candidates = fused_docs[:30]

        # Re-ranking
        cross_inp = [[query, doc.page_content] for doc in candidates]
        scores = self.reranker.predict(cross_inp)

        # จับคู่คะแนนกับเอกสาร แล้วเรียงลำดับใหม่
        scored_docs = list(zip(scores, candidates))
        sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)

        # เอาเฉพาะ top_k ส่งกลับไป
        return [doc for score, doc in sorted_docs[:top_k]]

    @staticmethod
    def reciprocal_rank_fusion(dense_docs, sparse_docs, k=60):
        doc_scores = {}
        for rank, d in enumerate(dense_docs):
            doc_scores[d.page_content] = doc_scores.get(d.page_content, 0) + 1 / (k + rank + 1)
        for rank, d in enumerate(sparse_docs):
            doc_scores[d.page_content] = doc_scores.get(d.page_content, 0) + 1 / (k + rank + 1)

        ranked = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        return [next(d for d in dense_docs + sparse_docs if d.page_content == content) for content, _ in ranked]

# Prompt แบบ Chain-of-Thought (CoT)
def build_prompt(context_docs: List[Document], question: str, choices: List[str]) -> str:
    context = "\n\n---\n\n".join([d.page_content for d in context_docs])
    choices_txt = "\n".join([f"{i+1}. {choices[i]}" for i in range(len(choices))])

    return f"""คุณคือ AI ผู้ช่วยอัจฉริยะของร้าน "ฟ้าใหม่ (FahMai)"
จงตอบคำถามโดยใช้ข้อมูลจาก [บริบท] ที่ให้มาเท่านั้น ห้ามเดาหรือใช้ความรู้ส่วนตัว

[บริบท]
{context}

[คำถาม]
{question}

[ตัวเลือก]
{choices_txt}

ให้คุณ "คิดทีละสเตป" (Chain-of-Thought) และเขียนอธิบายตามลำดับดังนี้:
1. คำถามนี้เป็นเรื่องที่เกี่ยวข้องกับสินค้า บริการ หรือข้อมูลของ "ร้านฟ้าใหม่" หรือไม่?
   - ถ้า "ไม่เกี่ยวข้องเลย" (เช่น ถามเรื่องทั่วไป, สภาพอากาศ, ร้านอื่น) -> ให้ตอบตัวเลือก 10 ทันที
   - ถ้า "เกี่ยวข้อง" ให้เขียนอธิบายสั้นๆ แล้วไปข้อ 2
2. ข้อมูลใน [บริบท] ที่ให้มา มีรายละเอียดเพียงพอที่จะหาคำตอบให้กับคำถามนี้หรือไม่? (ระวังเรื่องชื่อรุ่นคล้ายกัน หรือข้อมูลในตารางสเปค)
   - ถ้า "ไม่มีข้อมูลในบริบทเลย" หรือ "ข้อมูลมีไม่พอที่จะฟันธง" -> ให้ตอบตัวเลือก 9 ทันที
   - ถ้า "มีข้อมูลเพียงพอ" ให้เขียนอธิบายสั้นๆ แล้วไปข้อ 3
3. เทียบข้อมูลในบริบทกับตัวเลือกข้อ 1-8 ทีละข้อ และหาข้อที่ถูกต้องที่สุดเพียงข้อเดียว

*** ข้อควรระวัง: ห้ามเขียนแท็ก <answer> จนกว่าคุณจะวิเคราะห์ข้อ 1, 2 และ 3 เสร็จสิ้น ***

เมื่อวิเคราะห์ครบถ้วนแล้ว ให้สรุปคำตอบสุดท้ายเป็นตัวเลขเพียงข้อเดียว ใส่ไว้ในแท็ก <answer>ตัวเลข</answer> เท่านั้น

เริ่มการวิเคราะห์:"""

### โหลดเอกสารและเตรียม Retriever
เซลล์นี้จะเรียกใช้ฟังก์ชัน `load_markdown_documents` เพื่ออ่านและประมวลผลเอกสารทั้งหมด และสร้าง `HybridRetriever` ซึ่งเป็นหัวใจสำคัญในการค้นหาข้อมูลที่เกี่ยวข้องกับคำถาม

In [ ]:
# =========================================
# Data Loading & Retriever Initialization
# =========================================

print("Loading and chunking documents...")
docs = load_markdown_documents(BASE_PATH)

print("\nInitializing Hybrid Retriever...")
retriever = HybridRetriever(docs)

print("\n🎯 สร้าง Retriever เสร็จสมบูรณ์")

### รัน Inference และสร้างคำตอบ
เซลล์นี้จะวนลูปตอบคำถามแต่ละข้อ โดยใช้ `HybridRetriever` เพื่อดึงข้อมูลที่เกี่ยวข้อง, สร้าง Prompt และส่งไปให้ LLM ประมวลผลคำตอบ นอกจากนี้ยังมีการจัดการการเรียก API และบันทึกผลลัพธ์เป็น Checkpoint เพื่อให้สามารถรันต่อได้ในภายหลัง

In [ ]:
# =========================================
# Run Inference
# =========================================
import concurrent.futures
import pandas as pd
import re
from tqdm import tqdm
from openai import OpenAI
from datetime import datetime
from tenacity import retry, stop_after_attempt, wait_exponential
import json
import threading
import requests

TEST_MODE = False
CHECKPOINT_FILE = "./checkpoint_results.jsonl"

file_lock = threading.Lock()

def print_retry_warning(retry_state):
    print(f"\n⚠️ [API ล่ม/ติด Limit] กำลังรอ {retry_state.next_action.sleep} วินาที เพื่อ Retry ครั้งที่ {retry_state.attempt_number}...")

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    before_sleep=print_retry_warning
)
def call_thai_llm_turbo(prompt: str) -> str:
    headers = {
        "Content-Type": "application/json",
        "apikey": API_KEY
    }
    payload = {
        "model": "/model",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2048,
        "temperature": 0.01
    }
    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]

def process_single_question(row):
    qid = row["id"]
    question = row["question"]
    choices = [row[f"choice_{i}"] for i in range(1, 11)]

    retrieved_docs = retriever.retrieve(question, top_k=20)
    prompt = build_prompt(retrieved_docs, question, choices)

    # ยิง API
    response_text = call_thai_llm_turbo(prompt)

    # ดึงเฉพาะคำตอบสุดท้ายออกมา
    match = re.search(r'<answer>\s*(\d+)\s*</answer>', response_text)
    if match:
        answer = int(match.group(1))
        answer = 9 if answer < 1 or answer > 10 else answer
    else:
        answer = 9

    result = {
        "id": int(qid),
        "answer": answer,
        "question": question,
        "ai_reasoning": response_text
    }

    # เซฟลง JSONL ทันทีที่ทำเสร็จ 1 ข้อ
    with file_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    return result

# --- เริ่มต้นระบบรันต่อจากจุดเดิม ---
print("Loading questions...")
df_all = pd.read_csv(QUESTIONS_PATH)

if TEST_MODE:
    print("⚠️ รันในโหมดทดสอบ (Test Mode) - ทำแค่ 5 ข้อแรก")
    df_all = df_all.head(5)

# เช็กว่ามีของเก่าที่ทำค้างไว้ไหม
completed_ids = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                completed_ids.add(data["id"])
            except: continue
    print(f"🔄 พบข้อมูลเก่าที่ทำเสร็จแล้ว {len(completed_ids)} ข้อ จะเริ่มทำข้อที่เหลือ...")

# กรองเฉพาะข้อที่ยังไม่ได้ทำ
df_to_run = df_all[~df_all["id"].isin(completed_ids)]

if len(df_to_run) == 0:
    print("✅ ทำครบทุกข้อแล้ว!")
else:
    print(f"🚀 กำลังเริ่มรัน {len(df_to_run)} ข้อที่เหลือ...")
    results = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
          futures = {executor.submit(process_single_question, row): row for _, row in df_to_run.iterrows()}
          for future in tqdm(concurrent.futures.as_completed(futures), total=len(df_to_run), desc="🧠 AI กำลังตอบคำถาม"):
               results.append(future.result())

# --- รวมร่างและเซฟไฟล์ ---
final_results = []
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                final_results.append(json.loads(line))
            except: continue

if len(final_results) > 0:
    # เรียง ID และลบตัวซ้ำ
    final_df = pd.DataFrame(final_results).drop_duplicates(subset=['id']).sort_values("id")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # แยกเซฟ 2 ไฟล์
    # 1. ไฟล์ส่งประกวด (ตัดมาแค่ id กับ answer)
    submission_df = final_df[['id', 'answer']]
    sub_path = f"./submission_{timestamp}.csv"
    submission_df.to_csv(sub_path, index=False)

    # 2. ไฟล์สำหรับ Error Analysis (เก็บข้อความ CoT เอาไว้ใช้อ่าน)
    log_path = f"./analysis_log_{timestamp}.csv"
    final_df.to_csv(log_path, index=False, encoding="utf-8-sig")

    print(f"\n✨ เสร็จสมบูรณ์!")
    print(f"📌 ไฟล์ส่ง Leaderboard: {sub_path}")
    print(f"🕵️‍♂️ ไฟล์วิเคราะห์ Error: {log_path}")
else:
    print("⚠️ ยังไม่มีผลลัพธ์ให้เซฟ")

### วิเคราะห์ผลลัพธ์เบื้องต้น
เซลล์นี้จะช่วยวิเคราะห์คำตอบที่ได้จาก LLM โดยเฉพาะคำถามที่ AI ตอบว่าไม่มีข้อมูล (ตัวเลือก 9) หรือไม่เกี่ยวข้อง (ตัวเลือก 10) รวมถึงระบุข้อที่ AI แสดงความลังเลในคำตอบ

In [ ]:
import pandas as pd

# โหลดไฟล์ Log ล่าสุดของคุณ (แก้ชื่อไฟล์ให้ตรงกับที่คุณอัปโหลดนะครับ)
log_df = pd.read_csv("./analysis_log_20260328_162236.csv")

# 1. ดูข้อที่ตอบ 9 (ไม่มีข้อมูล) หรือ 10 (ไม่เกี่ยวข้อง)
suspicious_9_10 = log_df[log_df['answer'].isin([9, 10])]

print(f"🚨 พบข้อที่ตอบ 9 หรือ 10 จำนวน: {len(suspicious_9_10)} ข้อ")
for _, row in suspicious_9_10.iterrows():
    print(f"\nข้อ {row['id']}: {row['question']}")
    print(f"คำตอบ: {row['answer']}")
    print(f"เหตุผลย่อๆ: {row['ai_reasoning'][:300]}...")
    print("-" * 50)

# 2. ดูข้อที่ AI บ่นว่า "ไม่พบข้อมูล" แต่ดันเดาตอบ 1-8 (Hallucination) หรือข้อที่มีความสับสน
keywords = ["ไม่พบข้อมูล", "ไม่ระบุ", "ไม่ชัดเจน", "อาจจะ", "ขัดแย้ง"]
uncertain_mask = log_df['ai_reasoning'].str.contains('|'.join(keywords), na=False)
# กรองเอาเฉพาะข้อที่ตอบ 1-8 แต่ดันมีคำพวกนี้โผล่มา
uncertain_df = log_df[uncertain_mask & (~log_df['answer'].isin([9, 10]))]

print(f"\n⚠️ พบข้อที่ AI ลังเล (ตอบ 1-8 แต่บอกว่าไม่ชัดเจน): {len(uncertain_df)} ข้อ")
print(uncertain_df[['id', 'answer']].to_string(index=False))